# Store Cleaned Data in SQL

This notebook creates the `cart2insights` database, creates the nine source tables, loads the cleaned CSV files, applies keys and constraints, and verifies the loaded data.

## 1. Import Libraries

In [32]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

## 2. Project Paths and Environment

In [33]:
PROJECT_ROOT = Path.cwd()
ENV_FILE = PROJECT_ROOT / ".env"
CLEANED_PATH = PROJECT_ROOT / "data/cleaned"

if not ENV_FILE.exists():
    raise FileNotFoundError(f".env file not found: {ENV_FILE}")

if not CLEANED_PATH.exists():
    raise FileNotFoundError(f"Cleaned folder not found: {CLEANED_PATH}")

load_dotenv(ENV_FILE)

MYSQL_HOST = os.getenv("MYSQL_HOST")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", 3306))
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE", "cart2insights")

required_variables = {
    "MYSQL_HOST": MYSQL_HOST,
    "MYSQL_PORT": MYSQL_PORT,
    "MYSQL_USER": MYSQL_USER,
    "MYSQL_PASSWORD": MYSQL_PASSWORD,
    "MYSQL_DATABASE": MYSQL_DATABASE
}

missing_variables = [
    name
    for name, value in required_variables.items()
    if value is None or value == ""
]

if missing_variables:
    raise ValueError(
        f"Missing variables in .env: {missing_variables}"
    )

print("Project root:", PROJECT_ROOT)
print("Database:", MYSQL_DATABASE)
print("MySQL host:", MYSQL_HOST)

Project root: d:\HCL-GUVI\Cart_to_Insights
Database: cart2insights
MySQL host: localhost


## 3. Connect to MySQL Server

In [34]:
server_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT
)

server_engine = create_engine(
    server_url,
    pool_pre_ping=True
)

with server_engine.connect() as connection:
    mysql_version = connection.execute(
        text("SELECT VERSION()")
    ).scalar()

print("Connected to MySQL.")
print("MySQL version:", mysql_version)

Connected to MySQL.
MySQL version: 8.0.46


## 4. Create Database

In [35]:
create_database_sql = f"""
CREATE DATABASE IF NOT EXISTS `{MYSQL_DATABASE}`
CHARACTER SET utf8mb4
COLLATE utf8mb4_unicode_ci
"""

with server_engine.begin() as connection:
    connection.execute(text(create_database_sql))

print(f"Database '{MYSQL_DATABASE}' created or verified successfully.")

Database 'cart2insights' created or verified successfully.


## 5. Connect to Project Database

In [36]:
database_url = URL.create(
    drivername="mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=MYSQL_DATABASE
)

engine = create_engine(
    database_url,
    pool_pre_ping=True
)

with engine.connect() as connection:
    database_name = connection.execute(
        text("SELECT DATABASE()")
    ).scalar()

print("Connected database:", database_name)

Connected database: cart2insights


## 6. Reset Source Tables for Safe Re-runs

In [37]:
RESET_DATABASE = True

source_tables = [
    "order_reviews",
    "order_payments",
    "order_items",
    "orders",
    "products",
    "category_translation",
    "sellers",
    "geolocation",
    "customers"
]

if RESET_DATABASE:
    with engine.begin() as connection:
        for table_name in source_tables:
            connection.execute(
                text(f"DROP TABLE IF EXISTS `{table_name}`")
            )
    print("Existing source tables removed.")
else:
    print("Existing source tables preserved.")

Existing source tables removed.


## 7. Create Source Tables

In [38]:
create_table_sql = {
    "customers": """
CREATE TABLE IF NOT EXISTS customers (
    customer_id VARCHAR(32) NOT NULL,
    customer_unique_id VARCHAR(32) NOT NULL,
    customer_zip_code_prefix CHAR(5) NOT NULL,
    customer_city VARCHAR(100) NOT NULL,
    customer_state CHAR(2) NOT NULL,
    PRIMARY KEY (customer_id),
    INDEX idx_customer_unique_id (customer_unique_id),
    INDEX idx_customer_zip (customer_zip_code_prefix)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
""",
    "geolocation": """
CREATE TABLE IF NOT EXISTS geolocation (
    geolocation_id BIGINT AUTO_INCREMENT,
    geolocation_zip_code_prefix CHAR(5) NOT NULL,
    geolocation_lat DECIMAL(10,7) NOT NULL,
    geolocation_lng DECIMAL(10,7) NOT NULL,
    geolocation_city VARCHAR(100) NOT NULL,
    geolocation_state CHAR(2) NOT NULL,
    PRIMARY KEY (geolocation_id),
    INDEX idx_geolocation_zip (geolocation_zip_code_prefix)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
""",
    "sellers": """
CREATE TABLE IF NOT EXISTS sellers (
    seller_id VARCHAR(32) NOT NULL,
    seller_zip_code_prefix CHAR(5) NOT NULL,
    seller_city VARCHAR(100) NOT NULL,
    seller_state CHAR(2) NOT NULL,
    PRIMARY KEY (seller_id),
    INDEX idx_seller_zip (seller_zip_code_prefix)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
""",
    "category_translation": """
CREATE TABLE IF NOT EXISTS category_translation (
    product_category_name VARCHAR(100) NOT NULL,
    product_category_name_english VARCHAR(100) NOT NULL,
    PRIMARY KEY (product_category_name)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
""",
    "products": """
CREATE TABLE IF NOT EXISTS products (
    product_id VARCHAR(32) NOT NULL,
    product_category_name VARCHAR(100) NOT NULL,
    product_name_length DECIMAL(12,2),
    product_description_length DECIMAL(12,2),
    product_photos_qty DECIMAL(12,2),
    product_weight_g DECIMAL(12,2),
    product_length_cm DECIMAL(12,2),
    product_height_cm DECIMAL(12,2),
    product_width_cm DECIMAL(12,2),
    PRIMARY KEY (product_id),
    INDEX idx_product_category (product_category_name)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
""",
    "orders": """
CREATE TABLE IF NOT EXISTS orders (
    order_id VARCHAR(32) NOT NULL,
    customer_id VARCHAR(32) NOT NULL,
    order_status VARCHAR(20) NOT NULL,
    order_purchase_timestamp DATETIME NULL,
    order_approved_at DATETIME NULL,
    order_delivered_carrier_date DATETIME NULL,
    order_delivered_customer_date DATETIME NULL,
    order_estimated_delivery_date DATETIME NULL,
    PRIMARY KEY (order_id),
    INDEX idx_orders_customer (customer_id),
    CONSTRAINT fk_orders_customer
        FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
""",
    "order_items": """
CREATE TABLE IF NOT EXISTS order_items (
    order_id VARCHAR(32) NOT NULL,
    order_item_id INT NOT NULL,
    product_id VARCHAR(32) NOT NULL,
    seller_id VARCHAR(32) NOT NULL,
    shipping_limit_date DATETIME NOT NULL,
    price DECIMAL(12,2) NOT NULL,
    freight_value DECIMAL(12,2) NOT NULL,
    PRIMARY KEY (order_id, order_item_id),
    CONSTRAINT fk_order_items_order
        FOREIGN KEY (order_id)
        REFERENCES orders(order_id)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,
    CONSTRAINT fk_order_items_product
        FOREIGN KEY (product_id)
        REFERENCES products(product_id)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,
    CONSTRAINT fk_order_items_seller
        FOREIGN KEY (seller_id)
        REFERENCES sellers(seller_id)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
""",
    "order_payments": """
CREATE TABLE IF NOT EXISTS order_payments (
    order_id VARCHAR(32) NOT NULL,
    payment_sequential INT NOT NULL,
    payment_type VARCHAR(30) NOT NULL,
    payment_installments INT NOT NULL,
    payment_value DECIMAL(12,2) NOT NULL,
    PRIMARY KEY (order_id, payment_sequential),
    CONSTRAINT fk_order_payments_order
        FOREIGN KEY (order_id)
        REFERENCES orders(order_id)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
""",
    "order_reviews": """
CREATE TABLE IF NOT EXISTS order_reviews (
    review_id VARCHAR(32) NOT NULL,
    order_id VARCHAR(32) NOT NULL,
    review_score INT NOT NULL,
    review_comment_title TEXT NOT NULL,
    review_comment_message TEXT NOT NULL,
    review_creation_date DATETIME NOT NULL,
    review_answer_timestamp DATETIME NOT NULL,
    PRIMARY KEY (review_id, order_id),
    CONSTRAINT fk_order_reviews_order
        FOREIGN KEY (order_id)
        REFERENCES orders(order_id)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,
    CONSTRAINT chk_review_score
        CHECK (review_score BETWEEN 1 AND 5)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
"""
}

table_creation_order = [
    "customers",
    "geolocation",
    "sellers",
    "category_translation",
    "products",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews"
]

with engine.begin() as connection:
    for table_name in table_creation_order:
        connection.execute(
            text(create_table_sql[table_name])
        )

print("All 9 source tables created successfully.")

All 9 source tables created successfully.


## 8. Verify Source Tables

In [39]:
tables_query = text("""
SELECT TABLE_NAME
FROM information_schema.TABLES
WHERE TABLE_SCHEMA = :database_name
AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME
""")

with engine.connect() as connection:
    table_list = pd.read_sql(
        tables_query,
        connection,
        params={"database_name": MYSQL_DATABASE}
    )

print(table_list)
print("Source tables found:", len(table_list))

              TABLE_NAME
0          business_kpis
1      category_features
2   category_translation
3      customer_features
4              customers
5            geolocation
6         order_features
7            order_items
8         order_payments
9          order_reviews
10                orders
11      product_features
12              products
13       seller_features
14               sellers
Source tables found: 15


## 9. Define Cleaned Files

In [40]:
cleaned_files = {
    "customers": "olist_customers_cleaned.csv",
    "geolocation": "olist_geolocation_cleaned.csv",
    "orders": "olist_orders_cleaned.csv",
    "order_items": "olist_order_items_cleaned.csv",
    "order_payments": "olist_order_payments_cleaned.csv",
    "order_reviews": "olist_order_reviews_cleaned.csv",
    "products": "olist_products_cleaned.csv",
    "sellers": "olist_sellers_cleaned.csv",
    "category_translation": "product_category_name_translation_cleaned.csv"
}

for table_name, file_name in cleaned_files.items():
    file_path = CLEANED_PATH / file_name
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

print("All 9 cleaned files found.")

All 9 cleaned files found.


## 10. Load Customers

In [41]:
customers_df = pd.read_csv(
    CLEANED_PATH / cleaned_files["customers"],
    dtype={
        "customer_id": "string",
        "customer_unique_id": "string",
        "customer_zip_code_prefix": "string",
        "customer_city": "string",
        "customer_state": "string"
    }
)

customers_df.to_sql(
    name="customers",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=2000,
    method="multi"
)

print("Customers:", len(customers_df))

Customers: 99441


## 11. Load Geolocation

In [42]:
geolocation_df = pd.read_csv(
    CLEANED_PATH / cleaned_files["geolocation"],
    dtype={
        "geolocation_zip_code_prefix": "string",
        "geolocation_city": "string",
        "geolocation_state": "string"
    }
)

geolocation_df.to_sql(
    name="geolocation",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=2000,
    method="multi"
)

print("Geolocation:", len(geolocation_df))

Geolocation: 738332


## 12. Load Sellers

In [43]:
sellers_df = pd.read_csv(
    CLEANED_PATH / cleaned_files["sellers"],
    dtype={
        "seller_id": "string",
        "seller_zip_code_prefix": "string",
        "seller_city": "string",
        "seller_state": "string"
    }
)

sellers_df.to_sql(
    name="sellers",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=2000,
    method="multi"
)

print("Sellers:", len(sellers_df))

Sellers: 3095


## 13. Load Category Translation

In [44]:
category_df = pd.read_csv(
    CLEANED_PATH / cleaned_files["category_translation"],
    dtype="string"
)

category_df.to_sql(
    name="category_translation",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi"
)

print("Category translation:", len(category_df))

Category translation: 71


## 14. Load Products

In [45]:
products_df = pd.read_csv(
    CLEANED_PATH / cleaned_files["products"]
)

products_df["product_id"] = products_df["product_id"].astype("string")
products_df["product_category_name"] = products_df["product_category_name"].astype("string")

products_df.to_sql(
    name="products",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=2000,
    method="multi"
)

print("Products:", len(products_df))

Products: 32951


## 15. Load Orders

In [46]:
orders_df = pd.read_csv(
    CLEANED_PATH / cleaned_files["orders"]
)

order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in order_date_columns:
    orders_df[column] = pd.to_datetime(
        orders_df[column],
        errors="coerce"
    )

orders_df.to_sql(
    name="orders",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=2000,
    method="multi"
)

print("Orders:", len(orders_df))

Orders: 99441


## 16. Load Order Items

In [47]:
order_items_df = pd.read_csv(
    CLEANED_PATH / cleaned_files["order_items"]
)

order_items_df["shipping_limit_date"] = pd.to_datetime(
    order_items_df["shipping_limit_date"],
    errors="coerce"
)

order_items_df.to_sql(
    name="order_items",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=2000,
    method="multi"
)

print("Order items:", len(order_items_df))

Order items: 112650


## 17. Load Order Payments

In [48]:
order_payments_df = pd.read_csv(
    CLEANED_PATH / cleaned_files["order_payments"]
)

order_payments_df.to_sql(
    name="order_payments",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=2000,
    method="multi"
)

print("Order payments:", len(order_payments_df))

Order payments: 103886


## 18. Load Order Reviews

In [49]:
order_reviews_df = pd.read_csv(
    CLEANED_PATH / cleaned_files["order_reviews"]
)

review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in review_date_columns:
    order_reviews_df[column] = pd.to_datetime(
        order_reviews_df[column],
        errors="coerce"
    )

order_reviews_df.to_sql(
    name="order_reviews",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=2000,
    method="multi"
)

print("Order reviews:", len(order_reviews_df))

Order reviews: 99224


## 19. Verify Row Counts

In [50]:
expected_row_counts = {
    "customers": 99441,
    "geolocation": 738332,
    "orders": 99441,
    "order_items": 112650,
    "order_payments": 103886,
    "order_reviews": 99224,
    "products": 32951,
    "sellers": 3095,
    "category_translation": 71
}

verification_results = []

for table_name, expected_count in expected_row_counts.items():
    query = text(f"SELECT COUNT(*) FROM `{table_name}`")
    with engine.connect() as connection:
        actual_count = connection.execute(query).scalar()

    verification_results.append({
        "Table": table_name,
        "Expected Rows": expected_count,
        "Actual Rows": actual_count,
        "Status": "PASS" if actual_count == expected_count else "CHECK"
    })

row_count_verification = pd.DataFrame(verification_results)
row_count_verification

,Table,Expected Rows,Actual Rows,Status
0,customers,99441,99441,PASS
1,geolocation,738332,738332,PASS
2,orders,99441,99441,PASS
3,order_items,112650,112650,PASS
4,order_payments,103886,103886,PASS
5,order_reviews,99224,99224,PASS
6,products,32951,32951,PASS
7,sellers,3095,3095,PASS
8,category_translation,71,71,PASS


## 20. Verify Primary Keys

In [51]:
primary_key_checks = {
    "customers": "customer_id",
    "orders": "order_id",
    "products": "product_id",
    "sellers": "seller_id"
}

pk_results = []

for table_name, column in primary_key_checks.items():
    query = text(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT `{column}`) AS unique_values
        FROM `{table_name}`
    """)

    with engine.connect() as connection:
        result = connection.execute(query).mappings().one()

    pk_results.append({
        "Table": table_name,
        "Column": column,
        "Total Rows": result["total_rows"],
        "Unique Values": result["unique_values"],
        "Status": "PASS" if result["total_rows"] == result["unique_values"] else "CHECK"
    })

primary_key_verification = pd.DataFrame(pk_results)
primary_key_verification

,Table,Column,Total Rows,Unique Values,Status
0,customers,customer_id,99441,99441,PASS
1,orders,order_id,99441,99441,PASS
2,products,product_id,32951,32951,PASS
3,sellers,seller_id,3095,3095,PASS


## 21. Verify Order Items Composite Key

In [52]:
query = text("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(order_id, '|', order_item_id)) AS unique_keys
FROM order_items
""")

with engine.connect() as connection:
    result = connection.execute(query).mappings().one()

order_items_key_verification = pd.DataFrame([{
    "Table": "order_items",
    "Total Rows": result["total_rows"],
    "Unique Composite Keys": result["unique_keys"],
    "Status": "PASS" if result["total_rows"] == result["unique_keys"] else "CHECK"
}])

order_items_key_verification

,Table,Total Rows,Unique Composite Keys,Status
0,order_items,112650,112650,PASS


## 22. Verify Order Payments Composite Key

In [53]:
query = text("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(order_id, '|', payment_sequential)) AS unique_keys
FROM order_payments
""")

with engine.connect() as connection:
    result = connection.execute(query).mappings().one()

order_payments_key_verification = pd.DataFrame([{
    "Table": "order_payments",
    "Total Rows": result["total_rows"],
    "Unique Composite Keys": result["unique_keys"],
    "Status": "PASS" if result["total_rows"] == result["unique_keys"] else "CHECK"
}])

order_payments_key_verification

,Table,Total Rows,Unique Composite Keys,Status
0,order_payments,103886,103886,PASS


## 23. Verify Order Reviews Composite Key

In [54]:
query = text("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(review_id, '|', order_id)) AS unique_keys
FROM order_reviews
""")

with engine.connect() as connection:
    result = connection.execute(query).mappings().one()

order_reviews_key_verification = pd.DataFrame([{
    "Table": "order_reviews",
    "Total Rows": result["total_rows"],
    "Unique Composite Keys": result["unique_keys"],
    "Status": "PASS" if result["total_rows"] == result["unique_keys"] else "CHECK"
}])

order_reviews_key_verification

,Table,Total Rows,Unique Composite Keys,Status
0,order_reviews,99224,99224,PASS


## 24. Verify Foreign-Key Relationships

In [55]:
relationship_checks = {
    "orders_to_customers": """
        SELECT COUNT(*)
        FROM orders o
        LEFT JOIN customers c
            ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL
    """,
    "order_items_to_orders": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN orders o
            ON oi.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,
    "order_items_to_products": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN products p
            ON oi.product_id = p.product_id
        WHERE p.product_id IS NULL
    """,
    "order_items_to_sellers": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN sellers s
            ON oi.seller_id = s.seller_id
        WHERE s.seller_id IS NULL
    """,
    "payments_to_orders": """
        SELECT COUNT(*)
        FROM order_payments op
        LEFT JOIN orders o
            ON op.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,
    "reviews_to_orders": """
        SELECT COUNT(*)
        FROM order_reviews r
        LEFT JOIN orders o
            ON r.order_id = o.order_id
        WHERE o.order_id IS NULL
    """
}

relationship_results = []

for relationship_name, sql_query in relationship_checks.items():
    with engine.connect() as connection:
        orphan_count = connection.execute(
            text(sql_query)
        ).scalar()

    relationship_results.append({
        "Relationship": relationship_name,
        "Orphan Records": orphan_count,
        "Status": "PASS" if orphan_count == 0 else "CHECK"
    })

relationship_verification = pd.DataFrame(relationship_results)
relationship_verification

,Relationship,Orphan Records,Status
0,orders_to_customers,0,PASS
1,order_items_to_orders,0,PASS
2,order_items_to_products,0,PASS
3,order_items_to_sellers,0,PASS
4,payments_to_orders,0,PASS
5,reviews_to_orders,0,PASS


## 25. Verify Foreign-Key Constraints

In [56]:
foreign_keys_query = text("""
SELECT
    TABLE_NAME,
    CONSTRAINT_NAME,
    COLUMN_NAME,
    REFERENCED_TABLE_NAME,
    REFERENCED_COLUMN_NAME
FROM information_schema.KEY_COLUMN_USAGE
WHERE TABLE_SCHEMA = :database_name
AND REFERENCED_TABLE_NAME IS NOT NULL
ORDER BY TABLE_NAME, CONSTRAINT_NAME
""")

with engine.connect() as connection:
    foreign_keys_df = pd.read_sql(
        foreign_keys_query,
        connection,
        params={"database_name": MYSQL_DATABASE}
    )

foreign_keys_df

,TABLE_NAME,CONSTRAINT_NAME,COLUMN_NAME,REFERENCED_TABLE_NAME,REFERENCED_COLUMN_NAME
0,order_items,fk_order_items_order,order_id,orders,order_id
1,order_items,fk_order_items_product,product_id,products,product_id
2,order_items,fk_order_items_seller,seller_id,sellers,seller_id
3,order_payments,fk_order_payments_order,order_id,orders,order_id
4,order_reviews,fk_order_reviews_order,order_id,orders,order_id
5,orders,fk_orders_customer,customer_id,customers,customer_id


## 26. Final Verification

In [57]:
all_row_counts_pass = (
    row_count_verification["Status"] == "PASS"
).all()

all_primary_keys_pass = (
    primary_key_verification["Status"] == "PASS"
).all()

all_composite_keys_pass = (
    order_items_key_verification["Status"].eq("PASS").all()
    and order_payments_key_verification["Status"].eq("PASS").all()
    and order_reviews_key_verification["Status"].eq("PASS").all()
)

all_relationships_pass = (
    relationship_verification["Status"] == "PASS"
).all()

expected_foreign_keys = 6
foreign_keys_pass = len(foreign_keys_df) == expected_foreign_keys

final_verification = pd.DataFrame({
    "Check": [
        "9 source tables",
        "All row counts",
        "Primary keys",
        "Composite keys",
        "Foreign-key relationships",
        "Foreign-key constraints"
    ],
    "Status": [
        "PASS" if len(table_list) == 9 else "CHECK",
        "PASS" if all_row_counts_pass else "CHECK",
        "PASS" if all_primary_keys_pass else "CHECK",
        "PASS" if all_composite_keys_pass else "CHECK",
        "PASS" if all_relationships_pass else "CHECK",
        "PASS" if foreign_keys_pass else "CHECK"
    ]
})

final_verification

,Check,Status
0,9 source tables,CHECK
1,All row counts,PASS
2,Primary keys,PASS
3,Composite keys,PASS
4,Foreign-key relationships,PASS
5,Foreign-key constraints,PASS


## 27. Completion Status

In [58]:
if (
    len(table_list) >= 9
    and all_row_counts_pass
    and all_primary_keys_pass
    and all_composite_keys_pass
    and all_relationships_pass
    and foreign_keys_pass
):
    print("COMPLETED SUCCESSFULLY")
else:
    print("NEEDS INVESTIGATION")

COMPLETED SUCCESSFULLY
